## Notebook17

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
import polars as pl
from plotnine import ggplot, aes
from polars import col as c

import funs

ub = "https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/"

In [ ]:
play = pl.read_csv(ub + "data/shakespeare_plays.csv")
char = pl.read_csv(ub + "data/shakespeare_characters.csv")
line = pl.read_csv(ub + "data/shakespeare_lines.csv.gz")

### Questions

Back to the Folger Shakespeare corpus from Notebook13. `play` has one row per play,
with a `play_id` and a `title`. `char` has one row per *character × play*, with a
`character_id`, the `play_id`, and a `name`. `line` has one row per printed line, with
the `play_id`, the `character_id` of the speaker, the `act`, the `scene`, a
`line_number` such as `"3.1.64"`, and the `text`.

In Notebook13 we found three things we could not fix yet: a table we wanted with one
column per act, cells that hold several speakers at once, and a `line_number` that
packs three numbers into one string. Today we have the tools for all three.

Unless a question says otherwise, just print the result of each block; do not save it
to a variable.

1. Start with a warm-up from Notebook13. Count the number of lines in each act of each
play, grouping by `play_id` and `act`, and sort by `act`.

This table is long: 185 rows, one per *play × act*.

2. Take the code from the previous question and add a pivot, so that there is one row
per play and one column per act. Use `index=c.play_id`, `on=c.act`, and
`values=c.n_lines`.

Now there are 37 rows and a column for each of the five acts. The sort by `act` before
the pivot is what puts the new columns in the order 1 to 5.

3. Take the code from the previous question and join `play` onto the result so that
each row has the full title of the play.

4. Now one play. Filter `line` to *Macbeth* (`"mac"`), keep only the speakers with at
least 60 lines in the play using `.filter(pl.len().over(c.character_id) >= 60)`, count
the lines for each `character_id` and `act`, and pivot so there is one row per speaker
and one column per act.

Some cells are `null`, because not every speaker talks in every act. Duncan speaks only
in act 1, since he is murdered in act 2. Lady Macbeth has nothing in act 5; remember
that, because we will see why at the end of the notebook.

5. Go back to the code from question 2 and add an unpivot to the end, with
`index=c.play_id`, `variable_name="act"`, and `value_name="n_lines"`.

We are back to 185 rows, one per *play × act*. The pivot and the unpivot undo each
other and no number changed along the way.

6. A wide table is hard to plot, because the acts are column names rather than a
column. Take the code from the previous question, filter to the four tragedies
`["ham", "mac", "oth", "lr"]`, and cast `act` back to an integer with
`c.act.cast(pl.Int64)`. Then draw a line plot of the number of lines against the act,
with one color for each play.

The cast is needed because `act` came back from the unpivot as text: it had been a
column name. *Hamlet* drops by more than half from act 1 to act 2, even though act 2 is
not a short act on the page. Keep that in mind too.

7. On to the packed speakers. Add a column `speaker` equal to
`c.character_id.str.split(" #")`, which turns each cell into a list of speakers. Filter
to *Macbeth* and to the rows where `character_id` contains `"#"`, and select
`character_id` and `speaker`.

These are the lines spoken by more than one character at once. Each cell of `speaker`
is now a list of names. For an ordinary line, the list has just one name in it.

8. Take the code from the previous question and add `.explode(c.speaker)` before the
filter. Then count the lines for each `speaker` instead of selecting columns.

In Notebook13 there were four witch rows: three witches and a combined cell with 21
lines. Now there are three witches, and each one is credited with the lines they speak
together.

9. Draw the twelve biggest parts in Shakespeare. Take the exploded table, count the
lines for each `speaker` and `play_id`, and keep the top 12. Join `char` to get each
character's real name, using both columns of the key:
`left_on=[c.speaker, c.play_id]` and `right_on=[c.character_id, c.play_id]`. Sort by
the number of lines and draw a bar plot with `geom_col` and `coord_flip`.

The join needs both columns because `character_id` alone is not a key of `char`.
Joining on `character_id` alone would copy Richard III's bar three times, once for each
play he appears in.

Notice who is missing: Falstaff, the biggest comic part Shakespeare wrote, is not on
the chart.

The last packed cell is `line_number`, which holds three numbers. Splitting it into
columns needs a list method we have not covered, so the block below is given. It adds
an integer column `num` holding the line's position within its scene.

In [ ]:
lines = (
    line
    .with_columns(num = c.line_number.str.split(".").list.get(2).cast(pl.Int64))
)

10. Filter `lines` to *Hamlet*, act 3, scene 1. Sort by `num` and select `num`,
`character_id`, and `text`.

The scene now sorts in the order the lines are spoken.

11. Every scene numbers its lines from 1, so the largest `num` in a scene tells you how
many lines the printed scene has. Group `lines` by `play_id`, `act`, and `scene`, and
compute `recorded` (the number of rows) and `highest` (the maximum of `num`). Add a
column `missing` equal to `highest - recorded` and sort from the most missing to the
least.

Lines are missing. The worst scene is *1 Henry IV* 2.4, numbered up to 545 with only 24
lines in the table. It is the tavern scene, and Falstaff speaks in prose. This table
holds only the verse: that is why Falstaff missed the chart, why Lady Macbeth's
sleepwalking scene (in prose) left her act 5 empty, and why *Hamlet* drops in act 2.

12. Last question, and it is a written one. A pivot can be undone by an unpivot, but a
`.group_by()` and `.agg()` cannot be undone. What does that tell you about the shape in
which a dataset should be stored?

**Answer**:
